# M2: review, edits, near duplicates and SFT
CPU notebook for an existing 10,000-row snapshot on Drive.
Run sections in order. Pause at each review checkpoint and fill the CSV yourself.
Use new output directories for every revision. Never overwrite previous artifacts.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os
import subprocess
import sys

REPO_DIR = Path('/content/kids-story')
ROOT = Path('/content/drive/MyDrive/kids-story-runs/m2/REPLACE_WITH_RUN')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/coder-fishing/KidStory.git', str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
os.environ['PYTHONPATH'] = str(REPO_DIR / 'src')
def run_module(module, *args):
    subprocess.run([sys.executable, '-m', module, *map(str, args)], check=True)


## 1. Normalize snapshot and export review sheets
The M1 split is provisional. Only the SFT split after final review is used for training.

In [ ]:
run_module('kids_story.data',
    '--input', ROOT / 'source/sample.jsonl',
    '--text-field', 'vi',
    '--output-dir', ROOT / 'normalized_v1')
RECORDS = ROOT / 'normalized_v1/processed.jsonl'
run_module('kids_story.data.m2', 'export-edits',
    '--input', RECORDS, '--csv', ROOT / 'text_edits_v1.csv')
run_module('kids_story.data.m2', 'export-review',
    '--input', RECORDS, '--output-dir', ROOT / 'metadata_v1')


## 2. Human edit checkpoint
Read name_candidates.csv and translation_candidates.csv from the EDA run.
Fill text_edits_v1.csv: before must occur exactly once (include sentence context).
Set kind=name/translation/text, after, reason, decision=apply/keep/reject,
reviewer and reviewed_at with timezone. Leave undecided rows pending.
For multiple edits to one story, duplicate the row and assign a unique edit_id.
Do not change source_id or record_hash. Each applied edit needs fresh story review.


In [ ]:
run_module('kids_story.data.m2', 'apply-edits',
    '--input', RECORDS, '--csv', ROOT / 'text_edits_v1.csv',
    '--output-dir', ROOT / 'edited_v1')


## 3. Human metadata checkpoint
Fill edited_v1/review/metadata_review.csv after reading the final story.
Read docs/M2_REVIEW_GUIDE.md. Keep pending until reviewed.
Text edits in this sheet require another export/import review round.


In [ ]:
run_module('kids_story.data.m2', 'import-review',
    '--input', ROOT / 'edited_v1/records.jsonl',
    '--csv', ROOT / 'edited_v1/review/metadata_review.csv',
    '--baseline', ROOT / 'edited_v1/review/review_baseline.json',
    '--output-dir', ROOT / 'approved_import_v1')
FINAL_RECORDS = ROOT / 'approved_import_v1/reviewed_records.jsonl'
run_module('kids_story.data.m2', 'near',
    '--input', FINAL_RECORDS, '--output-dir', ROOT / 'near_final_v1')


## 4. Human near-duplicate checkpoint
Read each candidate pair. Set decision=confirmed/distinct and reviewer,
reviewed_at, review_notes. Pending pairs block SFT. No candidates does not prove
semantic uniqueness. Export only approved rows after resolving these decisions.


In [ ]:
run_module('kids_story.data.m2', 'sft',
    '--input', FINAL_RECORDS,
    '--near-report', ROOT / 'near_final_v1/near_candidates.json',
    '--near-review', ROOT / 'near_final_v1/near_review.csv',
    '--rights', REPO_DIR / 'data/review/rights_ledger.json',
    '--output-dir', ROOT / 'sft_final_v1',
    '--seed', 42, '--validation-ratio', 0.1)


## 5. Continue to M4
Set SFT_DIR in m4_qlora_colab.ipynb to sft_final_v1 above. The pool has 10,000
candidates; actual approved train/validation counts are recorded in manifest.json.
Rubric feedback and independent final holdout review remain human tasks.
